<a href="https://colab.research.google.com/github/letyresina/auditoria-calcadas-yolov8/blob/main/auditoria_calcadas_yolov8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Puxando as imagens para usar no CP

In [ ]:
import os
import time
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed


# ============================================================
# CONFIGURAÇÕES
# ============================================================

# Cole aqui seu token do Mapillary
ACCESS_TOKEN = "COLOQUE_SEU_TOKEN_AQUI"

# Usuário usado pelo projeto original
CREATOR_NAME = "gabinete_falzoni"

# ID do usuário usado no script R original
TARGET_USER_ID = "982503570396388"

# Mesmo período usado atualmente pelo script do repositório
START_DATE = "2025-11-08T00:00:00Z"
END_DATE   = "2025-11-17T23:59:59Z"

# Bounding box do Brás usada no repositório:
# xmin, ymin, xmax, ymax
BBOX = (
    -46.628036,
    -23.549392,
    -46.605377,
    -23.529917
)

# Pasta onde tudo será salvo
OUTPUT_DIR = "auditoria_calcadas"

IMAGES_DIR = os.path.join(OUTPUT_DIR, "imagens")
CSV_PATH = os.path.join(OUTPUT_DIR, "fotos_mapillary.csv")

# Número de divisões da área.
#
# O código original usa 100 x 100.
# Isso gera 10.000 pequenas consultas.
#
# Para começar, recomendo 20 x 20.
# Se perceber que faltaram imagens, aumente.
GRID_X = 20
GRID_Y = 20

# Downloads simultâneos
DOWNLOAD_WORKERS = 10

# Timeout HTTP
TIMEOUT = 30


# ============================================================
# PREPARAÇÃO
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

session = requests.Session()

session.headers.update({
    "User-Agent": "auditoria-calcadas-downloader/1.0"
})


# ============================================================
# CRIAR GRID
# ============================================================

def make_grid(bbox, nx=20, ny=20):
    """
    Divide a bounding box em pequenos retângulos.

    bbox:
        xmin, ymin, xmax, ymax
    """

    xmin, ymin, xmax, ymax = bbox

    dx = (xmax - xmin) / nx
    dy = (ymax - ymin) / ny

    tiles = []

    for i in range(nx):

        for j in range(ny):

            x1 = xmin + i * dx
            y1 = ymin + j * dy

            x2 = x1 + dx
            y2 = y1 + dy

            tiles.append(
                (x1, y1, x2, y2)
            )

    return tiles


# ============================================================
# CONSULTAR MAPILLARY
# ============================================================

def fetch_tile(tile):
    """
    Consulta todas as imagens de uma pequena região.
    """

    bbox_string = ",".join(map(str, tile))

    url = "https://graph.mapillary.com/images"

    params = {
        "fields": (
            "id,"
            "computed_geometry,"
            "thumb_original_url,"
            "captured_at,"
            "creator"
        ),

        "bbox": bbox_string,

        "creator_username": CREATOR_NAME,

        "start_captured_at": START_DATE,

        "end_captured_at": END_DATE,

        "limit": 1000,

        "access_token": ACCESS_TOKEN
    }

    imagens = []

    while url:

        try:

            response = session.get(
                url,
                params=params,
                timeout=TIMEOUT
            )

            response.raise_for_status()

            data = response.json()

        except requests.RequestException as e:

            print(
                f"\nErro consultando tile {tile}: {e}"
            )

            return imagens


        registros = data.get("data", [])

        imagens.extend(registros)


        # paginação

        paging = data.get("paging", {})

        url = paging.get("next")


        # A URL seguinte já contém os parâmetros,
        # portanto eles não devem ser enviados novamente

        params = None


        if url:
            time.sleep(0.25)


    return imagens


# ============================================================
# EXTRAIR LAT/LON
# ============================================================

def extrair_coordenadas(imagem):

    geometry = imagem.get("computed_geometry")

    if not geometry:
        return None, None

    coords = geometry.get("coordinates")

    if not coords or len(coords) < 2:
        return None, None

    lon = coords[0]
    lat = coords[1]

    return lat, lon


# ============================================================
# BUSCAR TODAS AS IMAGENS
# ============================================================

def buscar_imagens():

    tiles = make_grid(
        BBOX,
        GRID_X,
        GRID_Y
    )

    print(
        f"Área dividida em {len(tiles)} regiões."
    )

    todas = {}

    for numero, tile in enumerate(tiles, start=1):

        print(
            f"\rConsultando região "
            f"{numero}/{len(tiles)}",
            end=""
        )

        imagens = fetch_tile(tile)

        for imagem in imagens:

            image_id = str(
                imagem["id"]
            )

            # dicionário elimina duplicatas
            todas[image_id] = imagem


    print()

    print(
        f"{len(todas)} imagens únicas encontradas."
    )

    return list(todas.values())


# ============================================================
# CONVERTER PARA DATAFRAME
# ============================================================

def criar_dataframe(imagens):

    dados = []

    for imagem in imagens:

        creator = imagem.get(
            "creator",
            {}
        )

        creator_id = str(
            creator.get("id", "")
        )

        # mantém a mesma filtragem do código R
        if (
            TARGET_USER_ID
            and creator_id != TARGET_USER_ID
        ):
            continue


        lat, lon = extrair_coordenadas(
            imagem
        )


        captured_at = imagem.get(
            "captured_at"
        )

        if captured_at:

            captured_datetime = pd.to_datetime(
                captured_at,
                unit="ms",
                utc=True
            )

        else:

            captured_datetime = None


        dados.append({

            "id":
                str(imagem.get("id")),

            "creator_id":
                creator_id,

            "creator_username":
                creator.get("username"),

            "captured_at":
                captured_at,

            "captured_datetime":
                captured_datetime,

            "latitude":
                lat,

            "longitude":
                lon,

            "url":
                imagem.get(
                    "thumb_original_url"
                ),

        })


    df = pd.DataFrame(dados)

    if not df.empty:

        df = (
            df
            .drop_duplicates(
                subset=["id"]
            )
            .sort_values(
                "captured_datetime"
            )
            .reset_index(drop=True)
        )

    return df


# ============================================================
# DOWNLOAD DA IMAGEM
# ============================================================

def baixar_imagem(row):

    image_id = str(row["id"])

    url = row["url"]

    if not url:
        return image_id, "sem URL"


    destino = os.path.join(
        IMAGES_DIR,
        f"{image_id}.jpg"
    )


    # não baixa novamente
    if os.path.exists(destino):

        return image_id, "já existe"


    try:

        response = session.get(
            url,
            timeout=TIMEOUT
        )

        response.raise_for_status()


        with open(
            destino,
            "wb"
        ) as arquivo:

            arquivo.write(
                response.content
            )


        return image_id, "ok"


    except Exception as e:

        return (
            image_id,
            f"erro: {e}"
        )


# ============================================================
# DOWNLOAD PARALELO
# ============================================================

def baixar_todas(df):

    total = len(df)

    baixadas = 0
    existentes = 0
    erros = 0


    with ThreadPoolExecutor(
        max_workers=DOWNLOAD_WORKERS
    ) as executor:


        tarefas = {

            executor.submit(
                baixar_imagem,
                row
            ): row["id"]

            for _, row in df.iterrows()

        }


        for numero, future in enumerate(
            as_completed(tarefas),
            start=1
        ):

            image_id, status = (
                future.result()
            )


            if status == "ok":

                baixadas += 1


            elif status == "já existe":

                existentes += 1


            else:

                erros += 1

                print(
                    f"\n[ERRO] "
                    f"{image_id}: "
                    f"{status}"
                )


            print(

                f"\rDownload "
                f"{numero}/{total} "

                f"| novas: "
                f"{baixadas} "

                f"| existentes: "
                f"{existentes} "

                f"| erros: "
                f"{erros}",

                end=""

            )


    print()


# ============================================================
# MAIN
# ============================================================

from google.colab import userdata
ACCESS_TOKEN = userdata.get('Mapillary_api')

def main():


    print(
        "\n=== AUDITORIA DE CALÇADAS ===\n"
    )


    print(
        "1. Buscando imagens no Mapillary..."
    )

    imagens = buscar_imagens()


    print(
        "\n2. Organizando metadados..."
    )

    df = criar_dataframe(
        imagens
    )


    print(
        f"{len(df)} imagens após filtragem."
    )


    # adiciona nome futuro do arquivo

    if not df.empty:

        df["arquivo"] = (
            df["id"]
            .astype(str)
            .apply(
                lambda x:
                os.path.join(
                    "imagens",
                    f"{x}.jpg"
                )
            )
        )


    # CSV
    df.to_csv(
        CSV_PATH,
        index=False,
        encoding="utf-8"
    )


    print(
        f"CSV salvo em: "
        f"{CSV_PATH}"
    )


    print(
        "\n3. Baixando imagens..."
    )


    baixar_todas(df)


    print(
        "\n=== FINALIZADO ==="
    )

    print(
        f"Imagens: "
        f"{IMAGES_DIR}"
    )

    print(
        f"Metadados: "
        f"{CSV_PATH}"
    )

    print(
        f"Total: "
        f"{len(df)} imagens"
    )


if __name__ == "__main__":

    main()


=== AUDITORIA DE CALÇADAS ===

1. Buscando imagens no Mapillary...
Área dividida em 400 regiões.
Consultando região 400/400
5558 imagens únicas encontradas.

2. Organizando metadados...
5558 imagens após filtragem.
CSV salvo em: auditoria_calcadas/fotos_mapillary.csv

3. Baixando imagens...
Download 5558/5558 | novas: 5558 | existentes: 0 | erros: 0

=== FINALIZADO ===
Imagens: auditoria_calcadas/imagens
Metadados: auditoria_calcadas/fotos_mapillary.csv
Total: 5558 imagens


In [ ]:
import shutil

# Create a zip archive of the OUTPUT_DIR
shutil.make_archive(
    base_name=OUTPUT_DIR, # The name of the archive (e.g., 'auditoria_calcadas.zip')
    format='zip',        # The archive format
    root_dir=OUTPUT_DIR  # The directory to archive
)

print(f"A pasta '{OUTPUT_DIR}' foi zipada para '{OUTPUT_DIR}.zip'")

A pasta 'auditoria_calcadas' foi zipada para 'auditoria_calcadas.zip'


# Selecionando arquivo

In [1]:

# ============================================================
# AUDITORIA DE CALÇADAS — PREPARAÇÃO DO DATASET
# Upload Roboflow + Extração + Conversão + Auditoria
# ============================================================

!pip -q install ultralytics pyyaml

from google.colab import files
from pathlib import Path
from collections import Counter
import yaml
import zipfile
import shutil
import torch

# ------------------------------------------------------------
# 1. Diretórios
# ------------------------------------------------------------

BASE = Path("/content/auditoria_yolo")
DATASET_ORIGINAL = BASE / "dataset_original"
DATASET_DETECTION = BASE / "dataset_detection"
DATA_YAML = BASE / "data_detection.yaml"

BASE.mkdir(parents=True, exist_ok=True)

print("GPU disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU indisponível. Ative T4 GPU no Colab.")

# ------------------------------------------------------------
# 2. Upload do ZIP do Roboflow
# ------------------------------------------------------------

print("\n📦 Selecione o ZIP exportado do Roboflow (YOLOv8):")
arquivos = files.upload()

zips = [
    (nome, conteudo)
    for nome, conteudo in arquivos.items()
    if nome.lower().endswith(".zip")
]

if len(zips) != 1:
    raise ValueError("Selecione exatamente um arquivo ZIP.")

nome_zip, conteudo_zip = zips[0]

# Recriar apenas as pastas geradas por esta etapa
for pasta in [DATASET_ORIGINAL, DATASET_DETECTION]:
    if pasta.exists():
        shutil.rmtree(pasta)
    pasta.mkdir(parents=True)

if DATA_YAML.exists():
    DATA_YAML.unlink()

caminho_zip = BASE / nome_zip
caminho_zip.write_bytes(conteudo_zip)

# ------------------------------------------------------------
# 3. Extração segura
# ------------------------------------------------------------

with zipfile.ZipFile(caminho_zip) as z:
    for item in z.infolist():
        destino = (DATASET_ORIGINAL / item.filename).resolve()

        if not destino.is_relative_to(DATASET_ORIGINAL.resolve()):
            raise ValueError("ZIP contém caminho inválido.")

    z.extractall(DATASET_ORIGINAL)

arquivos_yaml = list(DATASET_ORIGINAL.rglob("data.yaml"))

if len(arquivos_yaml) != 1:
    raise ValueError(
        f"Esperava 1 data.yaml, encontrei {len(arquivos_yaml)}."
    )

RAIZ = arquivos_yaml[0].parent

with open(arquivos_yaml[0], encoding="utf-8") as f:
    config = yaml.safe_load(f)

# ------------------------------------------------------------
# 4. Conferir classes
# ------------------------------------------------------------

nomes = config.get("names", [])

if isinstance(nomes, dict):
    classes = [
        str(nomes[k])
        for k in sorted(nomes, key=lambda x: int(x))
    ]
else:
    classes = [str(nome) for nome in nomes]

esperadas = {"buraco", "desnivel", "piso_tatil"}

print("\nClasses encontradas:", classes)

if len(classes) != 3 or set(classes) != esperadas:
    raise ValueError(
        f"Classes inesperadas: {classes}. "
        "Verifique o projeto no Roboflow."
    )

# ------------------------------------------------------------
# 5. Converter polígonos em bounding boxes
# ------------------------------------------------------------

resumo = Counter()
erros = []

for split in ["train", "valid", "test"]:
    origem = RAIZ / split
    destino = DATASET_DETECTION / split

    pasta_imagens = origem / "images"
    pasta_labels = origem / "labels"

    if not pasta_imagens.exists():
        raise FileNotFoundError(f"Pasta ausente: {pasta_imagens}")

    shutil.copytree(pasta_imagens, destino / "images")

    destino_labels = destino / "labels"
    destino_labels.mkdir(parents=True, exist_ok=True)

    imagens = [
        p for p in (destino / "images").iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
    ]

    for imagem in imagens:
        label_original = pasta_labels / f"{imagem.stem}.txt"
        label_destino = destino_labels / f"{imagem.stem}.txt"

        linhas_convertidas = []

        if label_original.exists():
            linhas = label_original.read_text(
                encoding="utf-8"
            ).splitlines()

            for numero, linha in enumerate(linhas, start=1):
                partes = linha.strip().split()

                if not partes:
                    continue

                try:
                    classe_id = int(partes[0])
                    coordenadas = [
                        float(valor) for valor in partes[1:]
                    ]

                    if not 0 <= classe_id < len(classes):
                        raise ValueError("ID de classe inválido")

                    if not all(
                        0 <= valor <= 1
                        for valor in coordenadas
                    ):
                        raise ValueError(
                            "Coordenadas fora do intervalo 0 a 1"
                        )

                    if len(coordenadas) == 4:
                        # Bounding box já no formato YOLO
                        xc, yc, w, h = coordenadas
                        resumo["caixas_originais"] += 1

                    elif (
                        len(coordenadas) >= 6
                        and len(coordenadas) % 2 == 0
                    ):
                        # Polígono -> bounding box
                        xs = coordenadas[0::2]
                        ys = coordenadas[1::2]

                        xmin, xmax = min(xs), max(xs)
                        ymin, ymax = min(ys), max(ys)

                        w = xmax - xmin
                        h = ymax - ymin

                        xc = (xmin + xmax) / 2
                        yc = (ymin + ymax) / 2

                        resumo["poligonos_convertidos"] += 1

                    else:
                        raise ValueError(
                            "Formato de anotação desconhecido"
                        )

                    if w <= 0 or h <= 0:
                        raise ValueError("Caixa sem área")

                    if (
                        xc - w / 2 < -1e-5
                        or xc + w / 2 > 1 + 1e-5
                        or yc - h / 2 < -1e-5
                        or yc + h / 2 > 1 + 1e-5
                    ):
                        raise ValueError("Caixa fora da imagem")

                    linhas_convertidas.append(
                        f"{classe_id} "
                        f"{xc:.6f} {yc:.6f} "
                        f"{w:.6f} {h:.6f}"
                    )

                    resumo[f"{split}_{classes[classe_id]}"] += 1

                except (ValueError, IndexError) as erro:
                    erros.append(
                        f"{label_original.name}, "
                        f"linha {numero}: {erro}"
                    )

        label_destino.write_text(
            "\n".join(linhas_convertidas)
            + ("\n" if linhas_convertidas else ""),
            encoding="utf-8"
        )

        resumo[f"{split}_imagens"] += 1

if erros:
    print("\n❌ Anotações com problemas:")
    for erro in erros[:20]:
        print("-", erro)

    raise ValueError("Corrija as anotações antes do treinamento.")

# ------------------------------------------------------------
# 6. Criar YAML definitivo
# ------------------------------------------------------------

config_final = {
    "path": str(DATASET_DETECTION),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 3,
    "names": classes
}

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        config_final,
        f,
        allow_unicode=True,
        sort_keys=False
    )

# ------------------------------------------------------------
# 7. Relatório
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("RELATÓRIO DO DATASET")
print("=" * 55)

print("Polígonos convertidos:", resumo["poligonos_convertidos"])
print("Caixas originais:", resumo["caixas_originais"])

for split in ["train", "valid", "test"]:
    print(f"\n{split.upper()}")
    print("Imagens:", resumo[f"{split}_imagens"])

    for classe in classes:
        print(
            f"  {classe}: "
            f"{resumo[f'{split}_{classe}']} objetos"
        )

for classe in classes:
    if resumo[f"train_{classe}"] == 0:
        raise ValueError(
            f"Classe {classe} ausente do treino."
        )

print("\n⚠️ Atenção: classes ausentes em validação/teste")
print("devem ser registradas como limitação da avaliação.")

print("\n✅ DATASET PREPARADO COM SUCESSO!")
print("Configuração:", DATA_YAML)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 5.8 MB/s eta 0:00:00
GPU disponível: True
GPU: Tesla T4

📦 Selecione o ZIP exportado do Roboflow (YOLOv8):


Saving auditoria-calcadas.v1i.yolov8.zip to auditoria-calcadas.v1i.yolov8.zip

Classes encontradas: ['buraco', 'desnivel', 'piso_tatil']

RELATÓRIO DO DATASET
Polígonos convertidos: 106
Caixas originais: 48

TRAIN
Imagens: 129
  buraco: 51 objetos
  desnivel: 21 objetos
  piso_tatil: 66 objetos

VALID
Imagens: 11
  buraco: 3 objetos
  desnivel: 0 objetos
  piso_tatil: 7 objetos

TEST
Imagens: 6
  buraco: 0 objetos
  desnivel: 2 objetos
  piso_tatil: 4 objetos

⚠️ Atenção: classes ausentes em validação/teste
devem ser registradas como limitação da avaliação.

✅ DATASET PREPARADO COM SUCESSO!
Configuração: /content/auditoria_yolo/data_detection.yaml


# Treinamento e avaliação do Yolov8

In [2]:

# ============================================================
# AUDITORIA DE CALÇADAS — TREINAMENTO YOLOv8
# Treinamento + Teste + Exportação dos resultados
# ============================================================

from pathlib import Path
from ultralytics import YOLO
from google.colab import files
import torch
import zipfile
import shutil

BASE = Path("/content/auditoria_yolo")
DATA_YAML = BASE / "data_detection.yaml"
RESULTADOS = BASE / "resultados"
NOME_TREINO = "yolov8n_calcadas"

PASTA_TREINO = RESULTADOS / NOME_TREINO
BEST_PT = PASTA_TREINO / "weights" / "best.pt"

if not DATA_YAML.exists():
    raise FileNotFoundError(
        "Execute primeiro a célula de preparação do dataset."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "Ative a T4 GPU em Ambiente de execução > "
        "Alterar tipo de ambiente de execução."
    )

print("🚀 GPU:", torch.cuda.get_device_name(0))
print("📂 Dataset:", DATA_YAML)

# ------------------------------------------------------------
# 1. Treinamento
# ------------------------------------------------------------

modelo = YOLO("yolov8n.pt")

modelo.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=512,
    batch=8,
    patience=12,
    device=0,
    project=str(RESULTADOS),
    name=NOME_TREINO,
    exist_ok=True,
    seed=42,
    plots=True
)

# ------------------------------------------------------------
# 2. Avaliação no conjunto de teste
# ------------------------------------------------------------

if not BEST_PT.exists():
    raise FileNotFoundError("O modelo best.pt não foi gerado.")

melhor_modelo = YOLO(str(BEST_PT))

metricas = melhor_modelo.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=512,
    device=0,
    plots=True,
    project=str(RESULTADOS),
    name="avaliacao_teste",
    exist_ok=True
)

print("\n📊 RESULTADOS NO TESTE")
print(f"mAP@50: {metricas.box.map50:.4f}")
print(f"mAP@50-95: {metricas.box.map:.4f}")

# ------------------------------------------------------------
# 3. Salvar resultados
# ------------------------------------------------------------

ZIP_SAIDA = BASE / "modelo_auditoria_calcadas.zip"

with zipfile.ZipFile(
    ZIP_SAIDA, "w", zipfile.ZIP_DEFLATED
) as z:

    z.write(BEST_PT, arcname="best.pt")
    z.write(DATA_YAML, arcname="data.yaml")

    for pasta in [
        PASTA_TREINO,
        RESULTADOS / "avaliacao_teste"
    ]:
        if pasta.exists():
            for arquivo in pasta.rglob("*"):
                if arquivo.is_file() and arquivo != BEST_PT:
                    z.write(
                        arquivo,
                        arcname=str(
                            arquivo.relative_to(RESULTADOS)
                        )
                    )

print("\n✅ TREINAMENTO FINALIZADO!")
print("📦 Arquivo:", ZIP_SAIDA)

files.download(str(ZIP_SAIDA))


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
🚀 GPU: Tesla T4
📂 Dataset: /content/auditoria_yolo/data_detection.yaml
Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/auditoria_yolo/data_detection.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, er

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>